# Part 1: Your First Foundry Agent

In [1]:
# --- Reconnect to Foundry (run this first after opening a new notebook) ---
import subprocess, sys, json, shutil
from pathlib import Path

def run_cmd(args, check=True):
    exe = shutil.which(args[0])
    if exe: args = [exe] + args[1:]
    result = subprocess.run(args, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result

SUFFIX = run_cmd(["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"]).stdout.strip()[:6]
RESOURCE_GROUP = f"rg-foundry-workshop-{SUFFIX}"

outputs = json.loads(run_cmd([
    "az", "deployment", "group", "show", "-g", RESOURCE_GROUP, "-n", "main",
    "--query", "properties.outputs", "-o", "json"
]).stdout)

PROJECT_ENDPOINT = outputs["projectEndpoint"]["value"]
MODEL_DEPLOYMENT_NAME = outputs["modelDeploymentName"]["value"]
APP_INSIGHTS_CONN_STR = outputs["appInsightsConnectionString"]["value"]

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool, WebSearchPreviewTool, FunctionTool
from azure.identity import DefaultAzureCredential, AzureCliCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()
print(f"✅ Connected to: {PROJECT_ENDPOINT}")
print(f"   Model: {MODEL_DEPLOYMENT_NAME}")

✅ Connected to: https://fndry-ws-c9db47.services.ai.azure.com/api/projects/workshop-project
   Model: gpt-4.1


---
## Section 1: Your First Foundry Agent (~8 min)

**Foundry Prompt Agents** are server-side agent definitions that combine:
- A **model deployment** (e.g. gpt-4.1)
- **Instructions** (system prompt)
- **Tools** (function calls, web search, file search, etc.)
- **Versioning** — every `create_version()` creates an immutable version

Agents are accessed via the **Responses API** — the same OpenAI-compatible API you know,
but with an `agent_reference` that tells Foundry which agent to use.

In [2]:
# --- Create your first Foundry agent ---
first_agent = project_client.agents.create_version(
    agent_name="workshop-assistant",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a helpful workshop assistant. "
            "Answer questions concisely and clearly. "
            "If you don't know something, say so."
        ),
    ),
)
print(f"✅ Agent created: {first_agent.name} (version {first_agent.version})")


✅ Agent created: workshop-assistant (version 1)


In [3]:
# --- Chat with your agent via the Responses API ---
response = openai_client.responses.create(
    input="What is Microsoft Foundry and how does it help with AI development?",
    extra_body={
        "agent_reference": {"name": first_agent.name, "type": "agent_reference"}
    },
)
print(f"🧑 What is Microsoft Foundry and how does it help with AI development?\n")
print(f"🤖 {response.output_text}")


🧑 What is Microsoft Foundry and how does it help with AI development?

🤖 **Microsoft Foundry** is a program or initiative (not to be confused with any specific product) run by Microsoft to help accelerate digital innovation inside and outside the company. In different contexts, "Foundry" has referred to teams or labs at Microsoft focused on experimental development, rapid prototyping, and supporting startups or internal innovation.

**How it helps with AI development:**

1. **Rapid Prototyping:** Foundry teams often work on quickly building and validating new AI-powered solutions. This allows ideas to be tested and iterated rapidly, helping to identify what works before larger investments are made.

2. **Collaboration:** Microsoft Foundry typically brings together multidisciplinary teams, including engineers, data scientists, designers, and product managers. This environment fosters innovation and creative problem-solving in AI and related areas.

3. **Access to Tools and Resources:** 

#### 🔍 Hallucination vs. Grounded Response

The response above was likely **hallucinated** — gpt-4.1's training data predates Microsoft Foundry,
so it fabricates plausible-sounding but inaccurate details.

**Fix: Add an MCP (Model Context Protocol) server.** MCP lets agents call external tools
at inference time. By connecting the [Microsoft Learn docs](https://learn.microsoft.com) via
an MCP server, the agent retrieves **real documentation** instead of guessing.


In [4]:
# --- Grounding with MCP: Microsoft Learn Documentation ---
# Same question, but now the agent can search real docs via MCP.

from azure.ai.projects.models import MCPTool

grounded_agent = project_client.agents.create_version(
    agent_name="workshop-assistant-grounded",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a helpful workshop assistant. "
            "For any Microsoft related question, use the MCP tool to search Microsoft Learn documentation for accurate information. "
            "Always base your answer on the retrieved documentation and cite sources in that case."
        ),
        tools=[
            MCPTool(
                server_label="microsoft-learn",
                server_url="https://learn.microsoft.com/api/mcp",
                require_approval="never",
            ),
        ],
    ),
)
print(f"✅ Grounded agent created: {grounded_agent.name} (version {grounded_agent.version})")

# Ask the same question — now grounded in real documentation
response = openai_client.responses.create(
    input="What is Microsoft Foundry and how does it help with AI development?",
    extra_body={
        "agent_reference": {"name": grounded_agent.name, "type": "agent_reference"}
    },
)
print(f"\n🧑 What is Microsoft Foundry and how does it help with AI development?\n")
print(f"🤖 {response.output_text}")
print(f"\n✅ Compare this grounded response with the hallucinated one above!")
print(f"   The MCP server gave the agent access to live Microsoft Learn docs.")


✅ Grounded agent created: workshop-assistant-grounded (version 1)

🧑 What is Microsoft Foundry and how does it help with AI development?

🤖 **Microsoft Foundry** is a unified Azure platform-as-a-service for developing, deploying, and managing generative AI applications and Azure AI APIs. It brings together agents, models, and tools under one management structure, aimed at enabling enterprises, developers, and data scientists to focus on building intelligent solutions rather than managing infrastructure.

### How Microsoft Foundry Helps with AI Development

- **Unified AI Development Environment:** Foundry provides a simplified interface and code-first experiences for building, testing, deploying, and managing AI and machine learning solutions.
- **Comprehensive Tools:** It features extensive capabilities for data preparation, model deployment (Models as a Service—MaaS), model evaluation, monitoring, and version control.
- **Collaboration & Project Management:** Foundry offers shared wo

In [5]:
# --- Multi-turn conversation ---
# Use 'conversation' to maintain state across turns.
# The conversation object tracks history automatically — no need for previous_response_id.

conversation = openai_client.conversations.create()
print(f"Conversation ID: {conversation.id}\n")

# Turn 1
r1 = openai_client.responses.create(
    input="What are the main components of a Microsoft Foundry agent?",
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": grounded_agent.name, "type": "agent_reference"}},
)
print(f"🧑 Turn 1: What are the main components of a Foundry agent?")
print(f"🤖 {r1.output_text}\n")

# Turn 2 — follows up on the same conversation (context is automatic)
r2 = openai_client.responses.create(
    input="Can you give me a concrete example of the second component?",
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": grounded_agent.name, "type": "agent_reference"}},
)
print(f"🧑 Turn 2: Can you give me a concrete example of the second component?")
print(f"🤖 {r2.output_text}")


Conversation ID: conv_65affdafefaf18a300dDNpkbfyHx9ZrLf16jVj7y1WOiFvQMmO

🧑 Turn 1: What are the main components of a Foundry agent?
🤖 The main components of a Microsoft Foundry agent are:

1. **Model (LLM)**: The large language model (such as GPT-4o, Llama, DeepSeek, etc.) that provides reasoning and language capabilities for the agent.
2. **Instructions**: These define the agent’s goals, constraints, and behavior. In Foundry, instructions can be prompt-based (plain text instructions), workflow definitions, or custom code if you're building a hosted agent.
3. **Tools**: Various built-in and custom tools allow the agent to access and interact with data or perform actions. Examples include web search, file search, memory, code interpreters, and integration with external systems (via MCP servers).

In addition to these functional components, the Foundry Agent Service also provides key infrastructure elements:

- **Agent Runtime**: Hosts and scales both prompt and hosted agents, and manag

> 💡 **Check it out**: Open the [Foundry Portal](https://ai.azure.com) → your project → **Tracing**.
> You should see server-side traces for each `responses.create()` call with an `agent_reference`.


---
Proceed to `02-tools.ipynb`.